# Fuzzy Deep Learning Models — Evaluation & Benchmarking

## Imports & Settings

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================

# ---- standard library ----
import os
import random
import warnings

# ---- data & numeric ----
import numpy as np
import pandas as pd

# ---- plotting ----
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import rcParams

# ---- scikit-learn ----
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_curve, roc_auc_score
)
from sklearn.cluster import KMeans

# ---- deep learning ----
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# ---- reporting / statistics ----
from docx import Document
from scipy.stats import wilcoxon

# =========================================================
# SETTINGS
# =========================================================

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")
rcParams['font.family'] = 'Times New Roman'

seed = 42
np.random.seed(seed)
random.seed(seed)
torch.manual_seed(seed)

# =========================================================
# PATHS
# =========================================================
# Input data lives under DATA_DIR (override with: export DATA_DIR=/path/to/data)
from pathlib import Path
DATA_DIR = Path(os.environ.get("DATA_DIR", "../data"))

## Create Output Folders

In [ ]:
# =========================================================
# CREATE RESULT FOLDERS
# =========================================================

models = [
    "CNN",
    "WHFDL",
    "Original_FDNN",
    "FAE",
    "NF_RVFL"
]

base_dir = "../results"

os.makedirs(base_dir, exist_ok=True)

for model in models:
    os.makedirs(os.path.join(base_dir, model),exist_ok=True)

os.makedirs(os.path.join(base_dir, "Summary"),exist_ok=True)

print("Folders created successfully.")

## Load Data

In [ ]:
# =========================================================
# LOAD DATASETS
# =========================================================

dataset_files = {

    "MRMR": DATA_DIR / "mRMR_30f.csv"
}

splits = {}

for dataset_name, file in dataset_files.items():

    data = pd.read_csv(file)

    X = data.drop("label", axis=1)

    y = data["label"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

    splits[dataset_name] = {

        "X_train": X_train,
        "X_test": X_test,

        "y_train": y_train,
        "y_test": y_test,
    
        "X": X,
        "y": y
    }

print("Datasets loaded successfully.")

## Training & Evaluation Utilities

In [ ]:
# =========================================================
# DEEP LEARNING UTILITIES
# =========================================================

summary_results = []
cv_scores_storage = {}

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu")

print("Using device:", device)

# Model names whose forward() returns (logits, reconstruction) and whose
# loss = classification_loss + reconstruction_loss (autoencoder-based models)
DUAL_OUTPUT_MODELS = ("FAE",)

# =========================================================
# CREATE DATALOADER
# =========================================================

def create_dataloader(

    X_train,
    X_test,

    y_train,
    y_test,

    batch_size=32
):

    scaler = StandardScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
    X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
    y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

    train_dataset = TensorDataset(X_train_tensor, y_train_tensor)

    test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

    train_loader = DataLoader( train_dataset, batch_size=batch_size, shuffle=True)

    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


    return (

        train_loader,
        test_loader,

        X_train_tensor,
        X_test_tensor,
        y_test_tensor,

        scaler
    )

# =========================================================
# FUZZY LAYER INITIALIZATION HELPER
# =========================================================

def init_fuzzy_layer_if_needed(model, model_class, X_train_tensor):
    """K-Means-initializes a model's `.fuzzy` submodule, if it has one."""
    fuzzy_module = getattr(model, "fuzzy", None)
    if fuzzy_module is not None and hasattr(fuzzy_module, "init_from_data"):
        fuzzy_module.init_from_data(X_train_tensor)

# =========================================================
# DL TRAINING + EVALUATION FUNCTION
# =========================================================
def dl_cross_validation(
    model_class,
    X,
    y,
    epochs=20,
    batch_size=32,
    lr=0.001,
    n_splits=5
):

    skf = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=42
    )

    fold_accuracies = []

    for train_idx, val_idx in skf.split(X, y):

        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Create new model instance
        num_classes = len(np.unique(y))
        model = model_class(X.shape[1], num_classes).to(device)

        # Create dataloaders
        train_loader, _, X_train_tensor, X_val_tensor, y_val_tensor, _ = create_dataloader(
            X_train_fold,
            X_val_fold,
            y_train_fold,
            y_val_fold,
            batch_size
        )

        # K-Means initialization of the fuzzy branch (WHFDL only)
        init_fuzzy_layer_if_needed(model, model_class, X_train_tensor)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        # Train
        model.train()
        for epoch in range(epochs):
            for batch_X, batch_y in train_loader:
                batch_X = batch_X.to(device)
                batch_y = batch_y.to(device)

                optimizer.zero_grad()
                if model_class.__name__ in DUAL_OUTPUT_MODELS:
                    outputs, decoded = model(batch_X)
                    classification_loss = criterion(outputs, batch_y)
                    reconstruction_loss = F.mse_loss(decoded, batch_X)
                    loss = classification_loss + reconstruction_loss

                else:
                    outputs = model(batch_X)
                    loss = criterion(outputs, batch_y)

                loss.backward()
                optimizer.step()

        # Validate
        model.eval()
        with torch.no_grad():
            if model_class.__name__ in DUAL_OUTPUT_MODELS:
                outputs, _ = model(X_val_tensor.to(device))

            else:
                outputs = model(X_val_tensor.to(device))

            preds = torch.argmax(outputs, dim=1).cpu().numpy()

        acc = accuracy_score(y_val_fold, preds)
        fold_accuracies.append(acc)

    return fold_accuracies


def evaluate_dl_model(

    model_class,
    model_name,

    dataset_name,

    X_train,
    X_test,

    y_train,
    y_test,

    X_full,
    y_full,

    epochs=20,
    batch_size=32,
    learning_rate=0.001
):

    # =====================================================
    # CREATE RESULT FOLDER
    # =====================================================

    dataset_folder = (f"../results/{model_name}/{dataset_name}")

    os.makedirs(dataset_folder, exist_ok=True)

    # =====================================================
    # DATALOADER
    # =====================================================

    (
        train_loader,
        test_loader,

        X_train_tensor,
        X_test_tensor,
        y_test_tensor,

        scaler

    ) = create_dataloader(

        X_train,
        X_test,

        y_train,
        y_test,

        batch_size
    )

    # =====================================================
    # MODEL
    # =====================================================

    input_dim = X_train.shape[1]

    num_classes = len(np.unique(y_full))
    model = model_class(input_dim, num_classes).to(device)

    # K-Means initialization of the fuzzy branch (WHFDL only)
    init_fuzzy_layer_if_needed(model, model_class, X_train_tensor)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(

        model.parameters(),

        lr=learning_rate
    )

    # =====================================================
    # TRAINING
    # =====================================================

    model.train()

    for epoch in range(epochs):

        for batch_X, batch_y in train_loader:

            batch_X = batch_X.to(device)

            batch_y = batch_y.to(device)

            optimizer.zero_grad()

            # ==========================================
            # AUTOENCODER-BASED MODELS (dual loss)
            # ==========================================

            if model_name in DUAL_OUTPUT_MODELS:

                outputs, decoded = model(batch_X)

                classification_loss = criterion(outputs, batch_y)

                reconstruction_loss = F.mse_loss(decoded, batch_X)

                loss = (classification_loss + reconstruction_loss)

            else:

                outputs = model(batch_X)

                loss = criterion(outputs, batch_y)

            loss.backward()

            optimizer.step()

    # =====================================================
    # EVALUATION
    # =====================================================

    model.eval()

    with torch.no_grad():

        X_test_tensor = X_test_tensor.to(device)

        # ==============================================
        # AUTOENCODER-BASED MODELS (dual loss)
        # ==============================================

        if model_name in DUAL_OUTPUT_MODELS:

            outputs, _ = model(X_test_tensor)

        else:

            outputs = model(X_test_tensor)

        probabilities = torch.softmax(outputs, dim=1)[:,1]

        predictions = torch.argmax(outputs, dim=1)

        y_pred = predictions.cpu().numpy()

        y_prob = probabilities.cpu().numpy()

        y_true = y_test_tensor.numpy()

    # =====================================================
    # CLASSIFICATION REPORT
    # =====================================================

    report_dict = classification_report(y_true, y_pred, output_dict=True)
    accuracy = accuracy_score(y_true, y_pred)

    # =====================================================
    # PRECISION
    # =====================================================

    precision_class_0 = report_dict['0']['precision']

    precision_class_1 = report_dict['1']['precision']

    precision_macro = report_dict['macro avg']['precision']

    precision_weighted = report_dict['weighted avg']['precision']

    # =====================================================
    # RECALL
    # =====================================================

    recall_class_0 = report_dict['0']['recall']

    recall_class_1 = report_dict['1']['recall']

    recall_macro = report_dict['macro avg']['recall']

    recall_weighted = report_dict['weighted avg']['recall']

    # =====================================================
    # F1 SCORE
    # =====================================================

    f1_class_0 = report_dict['0']['f1-score']

    f1_class_1 = report_dict['1']['f1-score']

    f1_macro = report_dict['macro avg']['f1-score']

    f1_weighted = report_dict['weighted avg']['f1-score']

    # =====================================================
    # AUC
    # =====================================================

    auc_score = roc_auc_score(y_true, y_prob)

    # =====================================================
    # ROC
    # =====================================================
    fpr, tpr, _ = roc_curve(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(8, 8), dpi=300)

    ax.plot(fpr, tpr, color='blue', linewidth=2,
            label=f"ROC Curve (AUC = {auc_score:.6f})")
    ax.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random Guessing')
    
    ax.set_xlabel("False Positive Rate (FPR)", fontsize=14, fontweight='bold')
    ax.set_ylabel("True Positive Rate (TPR)", fontsize=14, fontweight='bold')
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal', adjustable='box')  # keep plot square
    
    ax.legend(prop={'size': 12}, loc='lower right')
    
    fig.savefig(f"{dataset_folder}/ROC_Curve.png", bbox_inches='tight')
    plt.close(fig)

    # =====================================================
    # CONFUSION MATRIX
    # =====================================================

    cm = confusion_matrix(y_test, y_pred)
    
    fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        vmin=0,
        vmax=100,
        xticklabels=['Negative', 'Positive'],
        yticklabels=['Negative', 'Positive'],
        annot_kws={'size': 14, 'weight': 'bold'},
        square=True,
        cbar=True,
        ax=ax
    )
    ax.set_xlabel("Predicted Label", fontweight='bold')
    ax.set_ylabel("True Label", fontweight='bold')
    fig.savefig(f"{dataset_folder}/Confusion_Matrix.png", bbox_inches='tight')
    plt.close(fig)


    # =====================================================
    # CROSS VALIDATION
    # =====================================================

    cv5_scores = dl_cross_validation(
        model_class,
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        lr=learning_rate,
        n_splits=5
    )
    
    cv10_scores = dl_cross_validation(
        model_class,
        X_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        lr=learning_rate,
        n_splits=10
    )

    cv5 = np.mean(cv5_scores)
    cv10 = np.mean(cv10_scores)

    cv_scores_storage[(dataset_name, model_name)] = cv10_scores
    
    # =====================================================
    # SUMMARY
    # =====================================================

    summary_results.append({

        "Dataset": dataset_name,
        "Model": model_name,

        # =========================================
        # PRECISION
        # =========================================

        "Precision_Class_0":
        precision_class_0,

        "Precision_Class_1":
        precision_class_1,

        "Precision_Macro_Avg":
        precision_macro,

        "Precision_Weighted_Avg":
        precision_weighted,

        # =========================================
        # RECALL
        # =========================================

        "Recall_Class_0":
        recall_class_0,

        "Recall_Class_1":
        recall_class_1,

        "Recall_Macro_Avg":
        recall_macro,

        "Recall_Weighted_Avg":
        recall_weighted,

        # =========================================
        # F1 SCORE
        # =========================================

        "F1_Class_0":
        f1_class_0,

        "F1_Class_1":
        f1_class_1,

        "F1_Macro_Avg":
        f1_macro,

        "F1_Weighted_Avg":
        f1_weighted,

        # =========================================
        # OTHER
        # =========================================
        "Accuracy_Macro_Avg": f1_macro,
        "Accuracy_weighted_Avg": f1_weighted,
        "Accuracy": accuracy,
        "AUC": auc_score,
        "CV5": cv5,
        "CV10": cv10
    })

    print(
        f"{model_name} | "
        f"{dataset_name} DONE")

## Model 1: CNN

In [ ]:
class CNN(nn.Module):

    def __init__(self, input_size, num_classes):

        super().__init__()

        self.conv1 = nn.Conv1d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = x.unsqueeze(1)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.pool(x).squeeze(-1)
        x = self.fc(x)
        return x

num_classes = len(np.unique(y))

for dataset_name, data in splits.items():
    evaluate_dl_model(
        model_class=CNN,
        model_name="CNN",
        dataset_name=dataset_name,

        X_train=data["X_train"],
        X_test=data["X_test"],

        y_train=data["y_train"],
        y_test=data["y_test"],

        X_full=data["X"],
        y_full=data["y"],

        epochs=20,
        batch_size=32,
        learning_rate=0.001)

## Model 2: WHFDL 

In [ ]:
class FuzzyMembership(nn.Module):
    """Gaussian fuzzy membership layer, K-Means initialized then fine-tuned by backprop (WHFDL paper)."""

    def __init__(self, input_dim, membership_units):
        super().__init__()
        self.membership_units = membership_units
        self.mu = nn.Parameter(torch.randn(membership_units, input_dim))
        self.log_sigma = nn.Parameter(torch.zeros(membership_units, input_dim))

    @torch.no_grad()
    def init_from_data(self, X):
        """K-Means initialization of (mu, sigma)."""
        X_np = X.detach().cpu().numpy()
        n_clusters = min(self.membership_units, len(X_np))
        km = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
        labels = km.fit_predict(X_np)

        centers = torch.zeros_like(self.mu)
        sigmas = torch.ones_like(self.mu)

        for k in range(self.membership_units):
            k_idx = k % n_clusters
            cluster_pts = X[torch.tensor(labels == k_idx)]
            centers[k] = torch.tensor(km.cluster_centers_[k_idx], dtype=torch.float32)
            if len(cluster_pts) > 1:
                sigmas[k] = cluster_pts.std(dim=0) + 1e-3
            else:
                sigmas[k] = X.std(dim=0) + 1e-3

        self.mu.copy_(centers)
        self.log_sigma.copy_(torch.log(sigmas))

    def forward(self, x):
        # x: (batch, input_dim) -> (batch, membership_units, input_dim)
        x = x.unsqueeze(1)
        sigma = torch.exp(self.log_sigma) + 1e-6
        membership = torch.exp(-((x - self.mu) ** 2) / (sigma ** 2))
        return membership


class WHFDL(nn.Module):
    """
    Hierarchical Fused Fuzzy Deep Neural Network. Fuzzy branch + deep branch -> weighted
    fusion -> classifier.
    """

    def __init__(
        self,
        input_size,
        membership_units=3,
        hidden_units=128,
        dropout_rate=0.3,
        num_classes=2
    ):
        super().__init__()

        # fuzzy branch
        self.fuzzy = FuzzyMembership(input_size, membership_units)
        self.fuzzy_proj = nn.Linear(membership_units, hidden_units)

        # deep branch
        self.deep_fc = nn.Linear(input_size, hidden_units)
        bound = 1.0 / np.sqrt(input_size)  # fan-in uniform init
        nn.init.uniform_(self.deep_fc.weight, -bound, bound)
        nn.init.zeros_(self.deep_fc.bias)

        # fusion block: trainable per-node weights for each branch
        self.w_d = nn.Parameter(torch.ones(hidden_units))
        self.w_f = nn.Parameter(torch.ones(hidden_units))
        self.fusion_bias = nn.Parameter(torch.zeros(hidden_units))

        self.fusion_fc = nn.Linear(hidden_units, hidden_units)
        self.dropout = nn.Dropout(dropout_rate)

        # output layer (softmax applied via CrossEntropyLoss / eval-time softmax)
        self.classifier = nn.Linear(hidden_units, num_classes)

    def forward(self, x):

        # fuzzy branch
        membership = self.fuzzy(x)                        # (B, M, D)
        fuzzy_rule = torch.prod(membership, dim=-1)        # (B, M)
        o_f = torch.sigmoid(self.fuzzy_proj(fuzzy_rule))   # (B, H)

        # deep branch
        o_d = torch.sigmoid(self.deep_fc(x))               # (B, H)

        # fusion + output
        fused = self.w_d * o_d + self.w_f * o_f + self.fusion_bias
        fused = torch.sigmoid(fused)
        fused = F.relu(self.fusion_fc(fused))
        fused = self.dropout(fused)

        out = self.classifier(fused)
        return out


for dataset_name, data in splits.items():

    evaluate_dl_model(
        model_class=WHFDL,
        model_name="WHFDL",
        dataset_name=dataset_name,

        X_train=data["X_train"],
        X_test=data["X_test"],

        y_train=data["y_train"],
        y_test=data["y_test"],

        X_full=data["X"],
        y_full=data["y"],

        epochs=20,
        batch_size=32,
        learning_rate=0.001)

## Model 3: FDNN

In [ ]:
class FuzzyMembershipSimple(nn.Module):
    """Gaussian membership function, randomly initialized (no K-Means)."""

    def __init__(self, input_dim, membership_units):
        super().__init__()
        self.mu = nn.Parameter(torch.randn(membership_units, input_dim))
        self.log_sigma = nn.Parameter(torch.zeros(membership_units, input_dim))

    def forward(self, x):
        x = x.unsqueeze(1)
        sigma = torch.exp(self.log_sigma) + 1e-6
        membership = torch.exp(-((x - self.mu) ** 2) / (sigma ** 2))
        return membership


class OriginalFDNN(nn.Module):
    """Plain fuzzy + dense baseline: one fuzzy layer concatenated with the raw input, no fusion or K-Means init."""

    def __init__(
        self,
        input_size,
        membership_units=3,
        dense_units=128,
        dropout_rate=0.4,
        num_classes=2
    ):
        super().__init__()

        self.membership = FuzzyMembershipSimple(input_size, membership_units)

        fused_size = input_size + membership_units

        self.fc1 = nn.Linear(fused_size, dense_units)
        self.dropout = nn.Dropout(dropout_rate)
        self.fc2 = nn.Linear(dense_units, num_classes)

    def forward(self, x):

        membership = self.membership(x)
        fuzzy_rule = torch.prod(membership, dim=-1)

        fused = torch.cat([x, fuzzy_rule], dim=1)

        x = F.relu(self.fc1(fused))
        x = self.dropout(x)
        x = self.fc2(x)

        return x


for dataset_name, data in splits.items():

    evaluate_dl_model(
        model_class=OriginalFDNN,
        model_name="Original_FDNN",
        dataset_name=dataset_name,

        X_train=data["X_train"],
        X_test=data["X_test"],

        y_train=data["y_train"],
        y_test=data["y_test"],

        X_full=data["X"],
        y_full=data["y"],

        epochs=20,
        batch_size=32,
        learning_rate=0.001)

## Model 4: FAE

In [ ]:
class FAE(nn.Module):
    """Autoencoder + fuzzy membership on the latent code. forward() returns (logits, reconstruction)."""

    def __init__(self, input_size, num_classes, latent_dim=16,
                 hidden_dim=64, membership_units=3):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_size, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_size)
        )

        self.membership = FuzzyMembershipSimple(latent_dim, membership_units)

        self.classifier = nn.Linear(latent_dim + membership_units, num_classes)

    def forward(self, x):
        z = self.encoder(x)
        decoded = self.decoder(z)

        membership = self.membership(z)
        fuzzy_rule = torch.prod(membership, dim=-1)

        fused = torch.cat([z, fuzzy_rule], dim=1)
        out = self.classifier(fused)

        return out, decoded


for dataset_name, data in splits.items():
    evaluate_dl_model(
        model_class=FAE,
        model_name="FAE",
        dataset_name=dataset_name,

        X_train=data["X_train"],
        X_test=data["X_test"],

        y_train=data["y_train"],
        y_test=data["y_test"],

        X_full=data["X"],
        y_full=data["y"],

        epochs=20,
        batch_size=32,
        learning_rate=0.001)

## Model 5: NF-RVFL

In [ ]:
class NF_RVFL(nn.Module):
    """Neuro-Fuzzy RVFL: frozen random enhancement layer + trainable fuzzy branch + direct input link -> output layer."""

    def __init__(self, input_size, num_classes,
                 enhancement_units=100, membership_units=3):
        super().__init__()

        # random, frozen enhancement layer
        self.random_layer = nn.Linear(input_size, enhancement_units)
        for p in self.random_layer.parameters():
            p.requires_grad = False

        self.membership = FuzzyMembershipSimple(input_size, membership_units)

        out_in = input_size + enhancement_units + membership_units
        self.classifier = nn.Linear(out_in, num_classes)

    def forward(self, x):

        enhanced = torch.tanh(self.random_layer(x))

        membership = self.membership(x)
        fuzzy_rule = torch.prod(membership, dim=-1)

        fused = torch.cat([x, enhanced, fuzzy_rule], dim=1)
        out = self.classifier(fused)
        return out


for dataset_name, data in splits.items():
    evaluate_dl_model(
        model_class=NF_RVFL,
        model_name="NF_RVFL",
        dataset_name=dataset_name,

        X_train=data["X_train"],
        X_test=data["X_test"],

        y_train=data["y_train"],
        y_test=data["y_test"],

        X_full=data["X"],
        y_full=data["y"],

        epochs=20,
        batch_size=32,
        learning_rate=0.001)

## Generate Summary Reports

In [ ]:
# =========================================================
# FINAL SUMMARY CELL
# =========================================================
summary_df = pd.DataFrame(summary_results)

datasets = summary_df["Dataset"].unique()

for dataset in datasets:

    dataset_folder = (f"../results/Summary/{dataset}")

    os.makedirs(dataset_folder, exist_ok=True)

    df_dataset = summary_df[summary_df["Dataset"] == dataset]

    def dataframe_to_docx(df, file_path, title):
        doc = Document()
        doc.add_heading(title, level=1)
    
        table = doc.add_table(rows=df.shape[0] + 1, cols=df.shape[1])
    
        # Add column headers
        for j, col in enumerate(df.columns):
            table.rows[0].cells[j].text = str(col)
    
        # Add data
        for i in range(df.shape[0]):
            for j in range(df.shape[1]):
                table.rows[i+1].cells[j].text = str(df.iloc[i, j])
    
        doc.save(file_path)

    # =====================================================
    # RECALL TABLE
    # =====================================================

    recall_table = df_dataset[["Model", "Recall_Class_0", "Recall_Class_1", "Recall_Macro_Avg", "Recall_Weighted_Avg"]].copy()

    cols_to_format = ["Recall_Class_0", "Recall_Class_1", "Recall_Macro_Avg", "Recall_Weighted_Avg"]
    recall_table[cols_to_format] = recall_table[cols_to_format].applymap(lambda x: f"{x:.2f}")

    dataframe_to_docx(recall_table, f"{dataset_folder}/Recall_Table.docx", "Recall Table")


    # =====================================================
    # PRECISION TABLE
    # =====================================================

    precision_table = df_dataset[["Model", "Precision_Class_0", "Precision_Class_1", "Precision_Macro_Avg", "Precision_Weighted_Avg"]].copy()
    
    cols = ["Precision_Class_0", "Precision_Class_1", "Precision_Macro_Avg", "Precision_Weighted_Avg"]
    
    precision_table[cols] = precision_table[cols].applymap(lambda x: f"{x:.2f}")
    
    dataframe_to_docx(precision_table, f"{dataset_folder}/Precision_Table.docx", "Precision Table")


    # =====================================================
    # F1 TABLE
    # =====================================================

    f1_table = df_dataset[["Model","F1_Class_0","F1_Class_1","F1_Macro_Avg","F1_Weighted_Avg"]].copy()
    
    cols = ["F1_Class_0", "F1_Class_1", "F1_Macro_Avg", "F1_Weighted_Avg"]
    
    f1_table[cols] = f1_table[cols].applymap(lambda x: f"{x:.2f}")
    
    dataframe_to_docx(f1_table, f"{dataset_folder}/F1_Table.docx", "F1 Table")

    # =====================================================
    # CV TABLE
    # =====================================================

    cv_table = df_dataset[["Model", "Accuracy", "Accuracy_Macro_Avg", "Accuracy_weighted_Avg", "CV5", "CV10"]].copy()
    
    cols = [
        "Accuracy",
        "Accuracy_Macro_Avg",
        "Accuracy_weighted_Avg",
        "CV5",
        "CV10",
    ]
    
    cv_table[cols] = cv_table[cols].applymap(lambda x: f"{x:.2f}")
    
    dataframe_to_docx(cv_table, f"{dataset_folder}/CV_Table.docx", "CV Table")
    
    # =====================================================
    # STATISTICAL TEST TABLE
    # =====================================================
    
    best_models = {
        "MRMR": "WHFDL"  
    }

    best_model_name = best_models[dataset]
    
    best_scores = cv_scores_storage[(dataset, best_model_name)]
    
    stats_rows = []
    
    for _, row in df_dataset.iterrows():
    
        current_model = row["Model"]
    
        # Skip comparison with itself
        if current_model == best_model_name:
    
            stats_rows.append({
                "Model": current_model,
                "Compared_With": "-",
                "Wilcoxon": "-",
                "P_Value": "-",
                "Z_Score": "-"
            })
    
        else:

            current_scores = cv_scores_storage[(dataset, current_model)]
    
            stat, p = wilcoxon(
                best_scores,
                current_scores
            )
    
            # ==========================================
            # Z SCORE APPROXIMATION
            # ==========================================
    
            n = len(best_scores)
    
            mean_w = n * (n + 1) / 4
    
            std_w = np.sqrt(
                n * (n + 1) * (2 * n + 1) / 24
            )
    
            z = (stat - mean_w) / std_w
    
            stats_rows.append({
    
                "Model": current_model,
    
                "Compared_With": best_model_name,
    
                "Wilcoxon": round(stat, 4),
    
                "P_Value": round(p, 6),
    
                "Z_Score": round(z, 4)
            })
    
    # ==========================================
    # CREATE DATAFRAME
    # ==========================================
    
    stats_table = pd.DataFrame(stats_rows)
    
    # ==========================================
    # SAVE TABLE
    # ==========================================
    
    dataframe_to_docx(
        stats_table,
        f"{dataset_folder}/Statistical_Test_Table.docx",
        "Statistical Test Table"
    )

print("ALL SUMMARIES CREATED SUCCESSFULLY")